In [1]:
# ==========================================
# CELL 1: IMPORTS, REPRODUCIBILITY & SETUP
# ==========================================
import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings('ignore')

# 1. Strict Reproducibility
SEED = 999
np.random.seed(SEED)
torch.manual_seed(SEED)

# 2. Output Directories (Strict Naming)
DIRS = ['results/tables', 'results/figures', 'results/models']
for d in DIRS:
    os.makedirs(d, exist_ok=True)

# 3. Hardware Optimization
GPU_AVAILABLE = torch.cuda.is_available()
MAX_WORKERS = min(os.cpu_count() - 1, 4)
DEVICE = 'cuda' if GPU_AVAILABLE else 'cpu'

print(f"✓ Seed fixed to {SEED}")
print(f"✓ GPU Acceleration: {GPU_AVAILABLE}")
print(f"✓ Parallel Workers: {MAX_WORKERS}")
print(f"✓ Clean Directories Created")

✓ Seed fixed to 999
✓ GPU Acceleration: True
✓ Parallel Workers: 4
✓ Clean Directories Created


In [2]:
# ==========================================
# CELL 2: CORE TRAINING ENGINE
# ==========================================
def optimized_trainer(X, y, task_type, is_bert):
    """Trains an XGBoost model using GPU DMatrix and StandardScaler"""
    # 1. Stratified Split 80/20
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=SEED, stratify=y
    )
    
    # 2. Normalization (Fit on train, transform both)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype('float32')
    X_test_scaled = scaler.transform(X_test).astype('float32')
    
    # Keep column names
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)
    
    # 3. Model Configuration
    objective = 'binary:logistic' if task_type == 'binary' else 'multi:softprob'
    eval_metric = ['logloss', 'auc'] if task_type == 'binary' else ['mlogloss', 'merror']
    
    params = {
        'objective': objective,
        'tree_method': 'hist',
        'device': DEVICE,
        'n_estimators': 2000,
        'early_stopping_rounds': 100,
        'learning_rate': 0.01, # Stable learning rate
        'max_depth': 10,
        'random_state': SEED,
        'n_jobs': MAX_WORKERS
    }
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=False)
    
    # 4. Predictions
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average=None, zero_division=0)
    
    return {
        'model': model, 'scaler': scaler, 'acc': acc, 
        'prec': prec, 'rec': rec, 'f1': f1,
        'y_test': y_test, 'y_pred': y_pred, 'features': X.columns
    }

In [3]:
# ==========================================
# CELL 3: TASK 1.1 - BASELINE & ARTIFACTS
# ==========================================
print("Loading Main Dataset...")
df = pd.read_csv('data/processed/processed_DB_articles.csv')
df = df.drop(columns=['Text'], errors='ignore')

# Separate Features
bert_cols = [c for c in df.columns if c.startswith('bert_')]
X_all = df.drop(columns=['is_AI', 'Writer'])
X_no_bert = X_all.drop(columns=bert_cols)

# Encoders
le_multi = LabelEncoder()
y_bin = df['is_AI']
y_multi = le_multi.fit_transform(df['Writer'])
multi_classes = list(le_multi.classes_)

# Parallel Training
tasks = {
    'Bin_NoBERT': (X_no_bert, y_bin, 'binary', False),
    'Bin_BERT': (X_all, y_bin, 'binary', True),
    'Multi_NoBERT': (X_no_bert, y_multi, 'multiclass', False),
    'Multi_BERT': (X_all, y_multi, 'multiclass', True)
}

results = {}
print("Starting Parallel Training for 4 Configurations...")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(optimized_trainer, *args): name for name, args in tasks.items()}
    for future in as_completed(futures):
        name = futures[future]
        results[name] = future.result()
        print(f"✓ {name} Completed - Acc: {results[name]['acc']:.4f}")
        # Save Models & Scalers
        joblib.dump(results[name]['model'], f"results/models/{name}_model.joblib")
        joblib.dump(results[name]['scaler'], f"results/models/{name}_scaler.joblib")

# --- GENERATE TABLES ---
# T1: Binary Results
t1_df = pd.DataFrame([
    {'Configuration': 'Features only', 'Accuracy': results['Bin_NoBERT']['acc'], 
     'Precision': results['Bin_NoBERT']['prec'].mean(), 'Recall': results['Bin_NoBERT']['rec'].mean(), 'F1-Score': results['Bin_NoBERT']['f1'].mean()},
    {'Configuration': 'Features + BERT', 'Accuracy': results['Bin_BERT']['acc'], 
     'Precision': results['Bin_BERT']['prec'].mean(), 'Recall': results['Bin_BERT']['rec'].mean(), 'F1-Score': results['Bin_BERT']['f1'].mean()}
])
t1_df.round(4).to_csv('results/tables/T1.csv', index=False)

# T2: Multi-class Results
t2_df = pd.DataFrame([
    {'Configuration': 'Features only', 'Accuracy': results['Multi_NoBERT']['acc'], 
     'Precision': results['Multi_NoBERT']['prec'].mean(), 'Recall': results['Multi_NoBERT']['rec'].mean(), 'F1-Score': results['Multi_NoBERT']['f1'].mean()},
    {'Configuration': 'Features + BERT', 'Accuracy': results['Multi_BERT']['acc'], 
     'Precision': results['Multi_BERT']['prec'].mean(), 'Recall': results['Multi_BERT']['rec'].mean(), 'F1-Score': results['Multi_BERT']['f1'].mean()}
])
t2_df.round(4).to_csv('results/tables/T2.csv', index=False)

# T3: Per-class F1 Multi-class
t3_df = pd.DataFrame({
    'Config': ['Features only', 'Features + BERT']
})
for i, cls in enumerate(multi_classes):
    t3_df[cls] = [results['Multi_NoBERT']['f1'][i], results['Multi_BERT']['f1'][i]]
t3_df.round(4).to_csv('results/tables/T3.csv', index=False)

# --- GENERATE FIGURES ---
def plot_cm(y_true, y_pred, labels, filename, title):
    plt.figure(figsize=(10, 8), dpi=300)
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.savefig(f'results/figures/{filename}', bbox_inches='tight')
    plt.close()

def plot_imp(model, features, filename, title):
    imp = pd.DataFrame({'Feature': features, 'Importance': model.feature_importances_}).sort_values('Importance', ascending=False).head(20)
    plt.figure(figsize=(12, 8), dpi=300)
    sns.barplot(x='Importance', y='Feature', data=imp, palette='viridis')
    plt.title(title)
    plt.savefig(f'results/figures/{filename}', bbox_inches='tight')
    plt.close()

# F1 & F2: Confusion Matrices (BERT)
plot_cm(results['Bin_BERT']['y_test'], results['Bin_BERT']['y_pred'], ['Human', 'AI'], 'F1.png', 'F1: Binary CM (Features+BERT)')
plot_cm(results['Multi_BERT']['y_test'], results['Multi_BERT']['y_pred'], multi_classes, 'F2.png', 'F2: Multi-class CM (Features+BERT)')

# F3, F4, F5, F6: Feature Importances
plot_imp(results['Bin_NoBERT']['model'], X_no_bert.columns, 'F3.png', 'F3: Binary Importance (Features Only)')
plot_imp(results['Bin_BERT']['model'], X_all.columns, 'F4.png', 'F4: Binary Importance (Features+BERT)')
plot_imp(results['Multi_NoBERT']['model'], X_no_bert.columns, 'F5.png', 'F5: Multi-class Importance (Features Only)')
plot_imp(results['Multi_BERT']['model'], X_all.columns, 'F6.png', 'F6: Multi-class Importance (Features+BERT)')

print("✓ Task 1.1 Complete. Tables T1-T3 and Figures F1-F6 saved.")

Loading Main Dataset...
Starting Parallel Training for 4 Configurations...


[20:47:51] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.



✓ Bin_NoBERT Completed - Acc: 0.9946
✓ Bin_BERT Completed - Acc: 0.9986
✓ Multi_NoBERT Completed - Acc: 0.7614
✓ Multi_BERT Completed - Acc: 0.8542
✓ Task 1.1 Complete. Tables T1-T3 and Figures F1-F6 saved.


In [4]:
# ==========================================
# CELL 4: TASK 1.2 - MISSPELLING SWEEP
# ==========================================
print("Running Misspelling Sweep...")
rates = [5, 10, 15, 20]
t4_data = []

# Baseline data (0%)
t4_data.append({
    'Perturbation %': '0% (baseline)',
    'Features F1': results['Bin_NoBERT']['f1'].mean(), 'Feat+BERT F1': results['Bin_BERT']['f1'].mean(),
    'Features Acc': results['Bin_NoBERT']['acc'], 'Feat+BERT Acc': results['Bin_BERT']['acc']
})

multi_acc_nobert, multi_acc_bert = [results['Multi_NoBERT']['acc']], [results['Multi_BERT']['acc']]

for rate in rates:
    df_miss = pd.read_csv(f'data/processed/processed_misspelled_{rate}.csv')
    df_miss = df_miss.drop(columns=['Text'], errors='ignore')
    
    X_miss_all = df_miss.drop(columns=['is_AI', 'Writer'])
    X_miss_nobert = X_miss_all.drop(columns=bert_cols)
    y_bin_miss = df_miss['is_AI']
    y_multi_miss = le_multi.transform(df_miss['Writer'])
    
    # Load Scalers & Transform
    X_miss_nobert_sc = pd.DataFrame(results['Bin_NoBERT']['scaler'].transform(X_miss_nobert), columns=X_miss_nobert.columns).astype('float32')
    X_miss_all_sc = pd.DataFrame(results['Bin_BERT']['scaler'].transform(X_miss_all), columns=X_miss_all.columns).astype('float32')
    
    # Binary Predictions
    pred_bin_nobert = results['Bin_NoBERT']['model'].predict(X_miss_nobert_sc)
    pred_bin_bert = results['Bin_BERT']['model'].predict(X_miss_all_sc)
    
    # Multi Predictions
    pred_multi_nobert = results['Multi_NoBERT']['model'].predict(X_miss_nobert_sc)
    pred_multi_bert = results['Multi_BERT']['model'].predict(X_miss_all_sc)
    
    multi_acc_nobert.append(accuracy_score(y_multi_miss, pred_multi_nobert))
    multi_acc_bert.append(accuracy_score(y_multi_miss, pred_multi_bert))
    
    t4_data.append({
        'Perturbation %': f'{rate}%',
        'Features F1': precision_recall_fscore_support(y_bin_miss, pred_bin_nobert, average='macro', zero_division=0)[2],
        'Feat+BERT F1': precision_recall_fscore_support(y_bin_miss, pred_bin_bert, average='macro', zero_division=0)[2],
        'Features Acc': accuracy_score(y_bin_miss, pred_bin_nobert),
        'Feat+BERT Acc': accuracy_score(y_bin_miss, pred_bin_bert)
    })

# T4 Table
pd.DataFrame(t4_data).round(4).to_csv('results/tables/T4.csv', index=False)

# F7 & F8: Degradation Curves
def plot_degradation(y1, y2, title, filename):
    plt.figure(figsize=(8, 6), dpi=300)
    x_labels = ['0%', '5%', '10%', '15%', '20%']
    plt.plot(x_labels, y1, marker='o', label='Features Only')
    plt.plot(x_labels, y2, marker='s', label='Features + BERT')
    plt.title(title)
    plt.ylabel('Accuracy')
    plt.xlabel('Perturbation Rate')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(f'results/figures/{filename}', bbox_inches='tight')
    plt.close()

plot_degradation([d['Features Acc'] for d in t4_data], [d['Feat+BERT Acc'] for d in t4_data], 'F7: Binary Misspelling Degradation', 'F7.png')
plot_degradation(multi_acc_nobert, multi_acc_bert, 'F8: Multi-class Misspelling Degradation', 'F8.png')

print("✓ Task 1.2 Complete. Table T4 and Figures F7, F8 saved.")

Running Misspelling Sweep...
✓ Task 1.2 Complete. Table T4 and Figures F7, F8 saved.


In [ ]:
# ==========================================
# CELL 5: TASK 1.3 - LENGTH SENSITIVITY
# ==========================================
print("Running Length Sensitivity Analysis...")
# Use original test set logic
df_full = pd.read_csv('data/processed/processed_DB_articles.csv')
_, df_test = train_test_split(df_full, test_size=0.20, random_state=SEED, stratify=df_full['is_AI'])

# Calculate word count and bin
df_test['Word_Count'] = df_test['Text'].astype(str).apply(lambda x: len(x.split()))
df_test['Length_Bin'] = pd.qcut(df_test['Word_Count'], q=3, labels=['Short', 'Medium', 'Long'])

t5_data = []
for bin_label in ['Short', 'Medium', 'Long']:
    bin_df = df_test[df_test['Length_Bin'] == bin_label]
    X_bin = bin_df.drop(columns=['Text', 'is_AI', 'Writer', 'Word_Count', 'Length_Bin'], errors='ignore')
    
    y_bin_true = bin_df['is_AI']
    y_multi_true = le_multi.transform(bin_df['Writer'])
    
    X_bin_nobert = X_bin.drop(columns=bert_cols)
    
    # Scale Data
    X_bin_nobert_sc = pd.DataFrame(results['Bin_NoBERT']['scaler'].transform(X_bin_nobert), columns=X_bin_nobert.columns).astype('float32')
    X_bin_all_sc = pd.DataFrame(results['Bin_BERT']['scaler'].transform(X_bin), columns=X_bin.columns).astype('float32')
    
    X_multi_nobert_sc = pd.DataFrame(results['Multi_NoBERT']['scaler'].transform(X_bin_nobert), columns=X_bin_nobert.columns).astype('float32')
    X_multi_all_sc = pd.DataFrame(results['Multi_BERT']['scaler'].transform(X_bin), columns=X_bin.columns).astype('float32')
    
    # Predict Binary
    p_bin_nobert = results['Bin_NoBERT']['model'].predict(X_bin_nobert_sc)
    p_bin_bert = results['Bin_BERT']['model'].predict(X_bin_all_sc)
    
    # Predict Multi-class
    p_multi_nobert = results['Multi_NoBERT']['model'].predict(X_multi_nobert_sc)
    p_multi_bert = results['Multi_BERT']['model'].predict(X_multi_all_sc)
    
    t5_data.append({
        'Length Bin': bin_label, 'N samples': len(bin_df),
        'Bin Features Acc': accuracy_score(y_bin_true, p_bin_nobert),
        'Bin Features F1': precision_recall_fscore_support(y_bin_true, p_bin_nobert, average='macro', zero_division=0)[2],
        'Bin Feat+BERT F1': precision_recall_fscore_support(y_bin_true, p_bin_bert, average='macro', zero_division=0)[2],
        'Multi Features Acc': accuracy_score(y_multi_true, p_multi_nobert),
        'Multi Features F1': precision_recall_fscore_support(y_multi_true, p_multi_nobert, average='macro', zero_division=0)[2],
        'Multi Feat+BERT F1': precision_recall_fscore_support(y_multi_true, p_multi_bert, average='macro', zero_division=0)[2]
    })

t5_df = pd.DataFrame(t5_data)
t5_df.round(4).to_csv('results/tables/T5.csv', index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=300)
x = np.arange(3)
width = 0.35

# Subplot 1: Binary
ax1.bar(x - width/2, t5_df['Bin Features F1'], width, label='Features Only', color='steelblue')
ax1.bar(x + width/2, t5_df['Bin Feat+BERT F1'], width, label='Features + BERT', color='skyblue')
ax1.set_xticks(x)
ax1.set_xticklabels(['Short', 'Medium', 'Long'])
ax1.set_title('Binary Detection F1 by Length', fontweight='bold')
ax1.set_ylabel('F1-Score')
ax1.set_ylim([0, 1.1])
ax1.legend()
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# Subplot 2: Multi-class
ax2.bar(x - width/2, t5_df['Multi Features F1'], width, label='Features Only', color='coral')
ax2.bar(x + width/2, t5_df['Multi Feat+BERT F1'], width, label='Features + BERT', color='lightsalmon')
ax2.set_xticks(x)
ax2.set_xticklabels(['Short', 'Medium', 'Long'])
ax2.set_title('Multi-class Attribution F1 by Length', fontweight='bold')
ax2.set_ylabel('F1-Score')
ax2.set_ylim([0, 1.1])
ax2.legend()
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('F9: Length Sensitivity Analysis', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('results/figures/F9.png', bbox_inches='tight')
plt.close()

print("✓ Task 1.3 Complete. Table T5 and Figure F9 saved.")

Running Length Sensitivity Analysis...
✓ Task 1.3 Complete. Table T5 and Figure F9 saved.


In [7]:
# ==========================================
# CELL 6: TASK 1.4 - CROSS DOMAIN
# ==========================================
print("Running Cross-Domain Evaluation...")
cd_files = {
    'MTG Essay (GPT4All vs Human)': 'data/processed/processed_cross_domain_essay.csv',
    'MTG WP (GPT4All vs Human)': 'data/processed/processed_cross_domain_wp.csv',
    'Reuters/Claude (unseen model)': 'data/processed/processed_unseen_reuters.csv'
}

t6_data = [{
    'Benchmark': 'Original test set',
    'Features Acc': results['Bin_NoBERT']['acc'], 'Feat+BERT Acc': results['Bin_BERT']['acc'],
    'Features F1': results['Bin_NoBERT']['f1'].mean(), 'Feat+BERT F1': results['Bin_BERT']['f1'].mean()
}]

train_features_nobert = results['Bin_NoBERT']['features']
train_features_all = results['Bin_BERT']['features']

for name, path in cd_files.items():
    df_cd = pd.read_csv(path)
    y_cd = df_cd['is_AI']
    
    X_cd_nobert = df_cd.reindex(columns=train_features_nobert, fill_value=0)
    X_cd_all = df_cd.reindex(columns=train_features_all, fill_value=0)
    
    # Scale
    X_nobert_sc = pd.DataFrame(results['Bin_NoBERT']['scaler'].transform(X_cd_nobert), columns=X_cd_nobert.columns).astype('float32')
    X_all_sc = pd.DataFrame(results['Bin_BERT']['scaler'].transform(X_cd_all), columns=X_cd_all.columns).astype('float32')
    
    # Predict
    p_nobert = results['Bin_NoBERT']['model'].predict(X_nobert_sc)
    p_bert = results['Bin_BERT']['model'].predict(X_all_sc)
    
    t6_data.append({
        'Benchmark': name,
        'Features Acc': accuracy_score(y_cd, p_nobert), 'Feat+BERT Acc': accuracy_score(y_cd, p_bert),
        'Features F1': precision_recall_fscore_support(y_cd, p_nobert, average='macro', zero_division=0)[2],
        'Feat+BERT F1': precision_recall_fscore_support(y_cd, p_bert, average='macro', zero_division=0)[2]
    })
    
    # F10 & F11 
    if 'Essay' in name:
        plot_cm(y_cd, p_bert, ['Human', 'AI'], 'F10_Essay.png', 'F10: Essay Benchmark CM')
    elif 'WP' in name:
        plot_cm(y_cd, p_bert, ['Human', 'AI'], 'F10_WP.png', 'F10: WP Benchmark CM')
    elif 'Reuters' in name:
        plot_cm(y_cd, p_bert, ['Human', 'AI'], 'F11.png', 'F11: Reuters Unseen CM')

pd.DataFrame(t6_data).round(4).to_csv('results/tables/T6.csv', index=False)
print("✓ Task 1.4 Complete. Table T6 and Figures F10, F11 saved.")

Running Cross-Domain Evaluation...
✓ Task 1.4 Complete. Table T6 and Figures F10, F11 saved.


In [ ]:
# ==========================================
# CELL 7: TASK 1.5 - ADVERSARIAL ATTACKS
# ==========================================
print("Running Adversarial Attacks...")
adv_files = {
    'Paraphrasing (Gemma-27b)': 'data/processed/processed_paraphrased_articles.csv',
    'Translation (NLLB)': 'data/processed/processed_translation.csv'
}

t7_data = [{
    'Attack': 'None (baseline)',
    'Features Acc': results['Bin_NoBERT']['acc'], 'Feat+BERT Acc': results['Bin_BERT']['acc'],
    'Features F1': results['Bin_NoBERT']['f1'].mean(), 'Feat+BERT F1': results['Bin_BERT']['f1'].mean()
}]

for name, path in adv_files.items():
    df_adv = pd.read_csv(path)
    
    # 1. BINARY EVALUATION
    y_adv = df_adv['is_AI']
    
    X_adv_nobert = df_adv.reindex(columns=train_features_nobert, fill_value=0)
    X_adv_all = df_adv.reindex(columns=train_features_all, fill_value=0)
    
    # Scale
    X_nobert_sc = pd.DataFrame(results['Bin_NoBERT']['scaler'].transform(X_adv_nobert), columns=X_adv_nobert.columns).astype('float32')
    X_all_sc = pd.DataFrame(results['Bin_BERT']['scaler'].transform(X_adv_all), columns=X_adv_all.columns).astype('float32')
    
    # Predict
    p_nobert = results['Bin_NoBERT']['model'].predict(X_nobert_sc)
    p_bert = results['Bin_BERT']['model'].predict(X_all_sc)
    
    t7_data.append({
        'Attack': name,
        'Features Acc': accuracy_score(y_adv, p_nobert), 'Feat+BERT Acc': accuracy_score(y_adv, p_bert),
        'Features F1': precision_recall_fscore_support(y_adv, p_nobert, average='macro', zero_division=0)[2],
        'Feat+BERT F1': precision_recall_fscore_support(y_adv, p_bert, average='macro', zero_division=0)[2]
    })
    
    # 2. MULTI-CLASS EVALUATION (F12)
    if 'Paraphrasing' in name and 'Writer' in df_adv.columns:
        df_adv_multi = df_adv.copy()
        
        df_adv_multi['Writer'] = df_adv_multi['Writer'].str.replace('_paraphrased', '')
        
        valid_idx = df_adv_multi['Writer'].isin(multi_classes)
        df_adv_multi = df_adv_multi[valid_idx] # Filtered copy just for multi-class
        
        if not df_adv_multi.empty:
            y_multi_adv = le_multi.transform(df_adv_multi['Writer'])
            X_adv_multi = df_adv_multi.reindex(columns=train_features_all, fill_value=0)
            
            X_multi_sc = pd.DataFrame(results['Multi_BERT']['scaler'].transform(X_adv_multi), columns=X_adv_multi.columns).astype('float32')
            p_multi_bert = results['Multi_BERT']['model'].predict(X_multi_sc)
            
            plot_cm(y_multi_adv, p_multi_bert, multi_classes, 'F12.png', 'F12: Paraphrasing Attack (Multi-class)')
            print("✓ Figure F12 (Paraphrasing CM) generated successfully.")
        else:
            print(f"⚠️ Note: No valid exact multi-class writers found in {name} to generate F12.")

# Add misspelling 20% from T4 (if available in notebook memory)
if 't4_data' in locals():
    t7_data.append({
        'Attack': 'Misspelling (20%)',
        'Features Acc': t4_data[-1]['Features Acc'], 'Feat+BERT Acc': t4_data[-1]['Feat+BERT Acc'],
        'Features F1': t4_data[-1]['Features F1'], 'Feat+BERT F1': t4_data[-1]['Feat+BERT F1']
    })

pd.DataFrame(t7_data).round(4).to_csv('results/tables/T7.csv', index=False)
print("✓ Task 1.5 Complete. Table T7 saved.")

Running Adversarial Attacks...
✓ Figure F12 (Paraphrasing CM) generated successfully.
✓ Task 1.5 Complete. Table T7 saved.


In [ ]:
# ==========================================
# CELL 8: TASK 1.6 - FEATURE ABLATION
# ==========================================
print("Running Feature Ablation Study (This will take a few minutes)...")
FEATURE_GROUPS = {
    "Perplexity + UID": ["ppl", "uid"],
    "Burstiness": ["burstiness"],
    "TTR + Stylometry": ["ttr", "complex", "avg_word_len", "avg_sent_len"],
    "LIWC/Empath": ['gain', 'beauty', 'government', 'urban', 'art', 'help', 'optimism', 'strength', 'love', 'traveling'],
    "Semantic Consistency": ["semantic_mean", "semantic_std"],
    "Syntax Depth": ["syntax_depth"],
    "BERT Embeddings": bert_cols
}

t8_data = [{
    'Removed Group': 'None (full model)',
    'Binary Acc': results['Bin_BERT']['acc'], 'Binary F1': results['Bin_BERT']['f1'].mean(),
    'Multi Acc': results['Multi_BERT']['acc'], 'Multi F1': results['Multi_BERT']['f1'].mean()
}]

# Base models (Full Features)
X_abl = df.drop(columns=['is_AI', 'Writer', 'Text'], errors='ignore')
y_bin_abl = df['is_AI']
y_multi_abl = le_multi.transform(df['Writer'])

for group, cols in FEATURE_GROUPS.items():
    print(f"  - Ablating {group}...")
    X_dropped = X_abl.drop(columns=cols, errors='ignore')
    
    # Train Binary
    bin_res = optimized_trainer(X_dropped, y_bin_abl, 'binary', True)
    # Train Multi
    mul_res = optimized_trainer(X_dropped, y_multi_abl, 'multiclass', True)
    
    t8_data.append({
        'Removed Group': group,
        'Binary Acc': bin_res['acc'], 'Binary F1': bin_res['f1'].mean(),
        'Multi Acc': mul_res['acc'], 'Multi F1': mul_res['f1'].mean()
    })

t8_df = pd.DataFrame(t8_data)
t8_df.round(4).to_csv('results/tables/T8.csv', index=False)

# F13: Horizontal Bar Chart (F1 Drop)
t8_df['Bin Drop'] = t8_df['Binary F1'].iloc[0] - t8_df['Binary F1']
t8_df['Multi Drop'] = t8_df['Multi F1'].iloc[0] - t8_df['Multi F1']
plot_df = t8_df.iloc[1:].sort_values('Bin Drop', ascending=True) # Ignore 'None'

plt.figure(figsize=(10, 8), dpi=300)
y_pos = np.arange(len(plot_df))
height = 0.35
plt.barh(y_pos - height/2, plot_df['Bin Drop'], height, label='Binary F1 Drop')
plt.barh(y_pos + height/2, plot_df['Multi Drop'], height, label='Multi-class F1 Drop')
plt.yticks(y_pos, plot_df['Removed Group'])
plt.xlabel('Decrease in F1-Score')
plt.title('F13: Feature Ablation Impact')
plt.legend()
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.savefig('results/figures/F13.png', bbox_inches='tight')
plt.close()

print("✓ Task 1.6 Complete. Table T8 and Figure F13 saved.")


Running Feature Ablation Study (This will take a few minutes)...
  - Ablating Perplexity + UID...
  - Ablating Burstiness...
  - Ablating TTR + Stylometry...
  - Ablating LIWC/Empath...
  - Ablating Semantic Consistency...
  - Ablating Syntax Depth...
  - Ablating BERT Embeddings...
✓ Task 1.6 Complete. Table T8 and Figure F13 saved.

🎉 ALL TASKS COMPLETED SUCCESSFULLY! Check the 'results/' folder. 🎉


In [ ]:
# ==========================================
# CELL 5: TASK 1.3 - LENGTH SENSITIVITY
# ==========================================
print("Running Length Sensitivity Analysis...")
# Use original test set logic
df_full = pd.read_csv('data/processed/processed_DB_articles.csv')
_, df_test = train_test_split(df_full, test_size=0.20, random_state=SEED, stratify=df_full['is_AI'])

# Calculate word count and bin
df_test['Word_Count'] = df_test['Text'].astype(str).apply(lambda x: len(x.split()))
df_test['Length_Bin'] = pd.qcut(df_test['Word_Count'], q=3, labels=['Short', 'Medium', 'Long'])

t5_data = []
for bin_label in ['Short', 'Medium', 'Long']:
    bin_df = df_test[df_test['Length_Bin'] == bin_label]
    X_bin = bin_df.drop(columns=['Text', 'is_AI', 'Writer', 'Word_Count', 'Length_Bin'], errors='ignore')
    
    y_bin_true = bin_df['is_AI']
    y_multi_true = le_multi.transform(bin_df['Writer'])
    
    X_bin_nobert = X_bin.drop(columns=bert_cols)
    
    # Scale Data
    X_bin_nobert_sc = pd.DataFrame(results['Bin_NoBERT']['scaler'].transform(X_bin_nobert), columns=X_bin_nobert.columns).astype('float32')
    X_bin_all_sc = pd.DataFrame(results['Bin_BERT']['scaler'].transform(X_bin), columns=X_bin.columns).astype('float32')
    
    X_multi_nobert_sc = pd.DataFrame(results['Multi_NoBERT']['scaler'].transform(X_bin_nobert), columns=X_bin_nobert.columns).astype('float32')
    X_multi_all_sc = pd.DataFrame(results['Multi_BERT']['scaler'].transform(X_bin), columns=X_bin.columns).astype('float32')
    
    # Predict Binary
    p_bin_nobert = results['Bin_NoBERT']['model'].predict(X_bin_nobert_sc)
    p_bin_bert = results['Bin_BERT']['model'].predict(X_bin_all_sc)
    
    # Predict Multi-class
    p_multi_nobert = results['Multi_NoBERT']['model'].predict(X_multi_nobert_sc)
    p_multi_bert = results['Multi_BERT']['model'].predict(X_multi_all_sc)
    
    t5_data.append({
        'Length Bin': bin_label, 'N samples': len(bin_df),
        'Bin Features Acc': accuracy_score(y_bin_true, p_bin_nobert),
        'Bin Features F1': precision_recall_fscore_support(y_bin_true, p_bin_nobert, average='macro', zero_division=0)[2],
        'Bin Feat+BERT F1': precision_recall_fscore_support(y_bin_true, p_bin_bert, average='macro', zero_division=0)[2],
        'Multi Features Acc': accuracy_score(y_multi_true, p_multi_nobert),
        'Multi Features F1': precision_recall_fscore_support(y_multi_true, p_multi_nobert, average='macro', zero_division=0)[2],
        'Multi Feat+BERT F1': precision_recall_fscore_support(y_multi_true, p_multi_bert, average='macro', zero_division=0)[2]
    })

t5_df = pd.DataFrame(t5_data)
t5_df.round(4).to_csv('results/tables/T5.csv', index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=300)
x = np.arange(3)
width = 0.35

# Subplot 1: Binary
ax1.bar(x - width/2, t5_df['Bin Features F1'], width, label='Features Only', color='steelblue')
ax1.bar(x + width/2, t5_df['Bin Feat+BERT F1'], width, label='Features + BERT', color='skyblue')
ax1.set_xticks(x)
ax1.set_xticklabels(['Short', 'Medium', 'Long'])
ax1.set_title('Binary Detection F1 by Length', fontweight='bold')
ax1.set_ylabel('F1-Score')
ax1.set_ylim([0, 1.1])
ax1.legend()
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# Subplot 2: Multi-class
ax2.bar(x - width/2, t5_df['Multi Features F1'], width, label='Features Only', color='coral')
ax2.bar(x + width/2, t5_df['Multi Feat+BERT F1'], width, label='Features + BERT', color='lightsalmon')
ax2.set_xticks(x)
ax2.set_xticklabels(['Short', 'Medium', 'Long'])
ax2.set_title('Multi-class Attribution F1 by Length', fontweight='bold')
ax2.set_ylabel('F1-Score')
ax2.set_ylim([0, 1.1])
ax2.legend()
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('F9: Length Sensitivity Analysis', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('results/figures/F9.png', bbox_inches='tight')
plt.close()

print("✓ Task 1.3 Complete. Table T5 and Figure F9 saved.")

Running Length Sensitivity Analysis...
✓ Task 1.3 Complete. Table T5 and Figure F9 saved.
